# Extract audio from TikTok videos (Top_100 & Normal_100)

Dieses Notebook geht rekursiv durch `tiktok_100_vs_100/Top_100/` und `tiktok_100_vs_100/Normal_100/`, extrahiert die Audiospur jedes `.mp4`-Videos und speichert sie als `.wav` in `data/processed/audio/` mit 44.1 kHz.

Es werden Fortschrittsanzeigen (`tqdm`) und ein Log (`audio_Analyse/logs/extraction.log`) angelegt. Am Ende wird eine kurze Zusammenfassung (Anzahl gesamt / erfolgreich / Fehler) ausgegeben.

Hinweis: `ffmpeg` muss auf dem System installiert und im PATH erreichbar sein (moviepy verwendet ffmpeg unter der Haube).

In [1]:
# Optional: uncomment to install python packages in the notebook environment (run only if needed)
# import sys, subprocess
# subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'moviepy', 'tqdm', 'librosa', 'soundfile', 'imageio-ffmpeg'])

# Note: ffmpeg itself must be installed separately (system package).

In [2]:
import logging
from pathlib import Path
from moviepy import VideoFileClip
from tqdm import tqdm
import traceback
import sys

# Robustly locate repository root (walk upwards looking for common markers)
def find_repo_root(start: Path = None):
    start = Path.cwd() if start is None else Path(start)
    markers = ['tiktok_100_vs_100', 'requirements.txt', 'README.md', '.git']
    for p in [start] + list(start.parents):
        for m in markers:
            if (p / m).exists():
                return p
    return start

repo_root = find_repo_root()
# Primary expected video folder at repo root; fall back to searching underneath repo_root
video_base = repo_root / 'tiktok_100_vs_100'
if not video_base.exists():
    candidates = list(repo_root.rglob('tiktok_100_vs_100'))
    if candidates:
        video_base = candidates[0]

out_dir = repo_root / 'data' / 'processed' / 'audio'
log_dir = repo_root / 'audio_Analyse' / 'logs'
out_dir.mkdir(parents=True, exist_ok=True)
log_dir.mkdir(parents=True, exist_ok=True)

# Logging setup
log_file = log_dir / 'extraction.log'
logger = logging.getLogger('audio_extraction')
logger.setLevel(logging.INFO)
# Clear existing handlers to avoid duplicates in notebooks
if logger.hasHandlers():
    logger.handlers.clear()
fh = logging.FileHandler(log_file, mode='a', encoding='utf-8')
formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
fh.setFormatter(formatter)
ch = logging.StreamHandler()
ch.setFormatter(formatter)
logger.addHandler(fh)
logger.addHandler(ch)

logger.info(f'Repo root: {repo_root}')
logger.info(f'Video base: {video_base}')
logger.info(f'Output audio dir: {out_dir}')
logger.info(f'Log file: {log_file}')

2025-11-06 08:27:21,243 - INFO - Repo root: c:\Users\CelinaB\Documents\FHDW\Semester 5\Projekt Algorithmen\Viralytics\Viralitaetsanalyse
2025-11-06 08:27:21,244 - INFO - Video base: c:\Users\CelinaB\Documents\FHDW\Semester 5\Projekt Algorithmen\Viralytics\Viralitaetsanalyse\tiktok_100_vs_100
2025-11-06 08:27:21,245 - INFO - Output audio dir: c:\Users\CelinaB\Documents\FHDW\Semester 5\Projekt Algorithmen\Viralytics\Viralitaetsanalyse\data\processed\audio
2025-11-06 08:27:21,245 - INFO - Log file: c:\Users\CelinaB\Documents\FHDW\Semester 5\Projekt Algorithmen\Viralytics\Viralitaetsanalyse\audio_Analyse\logs\extraction.log
2025-11-06 08:27:21,244 - INFO - Video base: c:\Users\CelinaB\Documents\FHDW\Semester 5\Projekt Algorithmen\Viralytics\Viralitaetsanalyse\tiktok_100_vs_100
2025-11-06 08:27:21,245 - INFO - Output audio dir: c:\Users\CelinaB\Documents\FHDW\Semester 5\Projekt Algorithmen\Viralytics\Viralitaetsanalyse\data\processed\audio
2025-11-06 08:27:21,245 - INFO - Log file: c:\Users

In [3]:
# Function: extract_audio_from_video (robust, closes resources, returns (ok, message))
import time

def extract_audio_from_video(video_path: Path, output_dir: Path, sr: int = 44100, overwrite: bool = False, retries: int = 1) -> (bool, str):
    """Extract audio from a single video and save as WAV (pcm_s16le).
    Returns (True, 'OK') or (False, 'error message')."""
    out_path = output_dir / f"{video_path.stem}.wav"
    if out_path.exists() and not overwrite:
        return True, f"Exists: {out_path.name}"
    attempt = 0
    while attempt <= retries:
        try:
            # Use VideoFileClip to access audio stream
            clip = VideoFileClip(str(video_path))
            if clip.audio is None:
                try: clip.reader.close()
                except Exception: pass
                return False, 'No audio track'
            # write_audiofile delegates to ffmpeg; ensure we use a conservative codec
            clip.audio.write_audiofile(str(out_path), fps=sr, codec='pcm_s16le')
            # try to close resources
            try: clip.reader.close()
            except Exception: pass
            try: clip.audio.reader.close_proc()
            except Exception: pass
            return True, 'OK'
        except Exception as e:
            tb = traceback.format_exc()
            msg = f"Error: {e}\n{tb}"
            attempt += 1
            if attempt > retries:
                return False, msg
            else:
                logger.warning(f'Retry {attempt}/{retries} for {video_path} after error: {e}')
                time.sleep(0.5)


In [4]:
# Extraction loop: find mp4s, extract audio, save summary CSV
from pathlib import Path
import pandas as pd

# Collect videos
video_paths = []
for sub in ['Top_100', 'Normal_100']:
    folder = video_base / sub
    if folder.exists():
        video_paths.extend(sorted(list(folder.rglob('*.mp4'))))
    else:
        logger.warning(f'Subfolder not found: {folder}')

total = len(video_paths)
logger.info(f'Found {total} video files to process')

results = []
for vp in tqdm(video_paths, desc='Extracting audio'):
    ok, msg = extract_audio_from_video(vp, out_dir, sr=44100, overwrite=False, retries=1)
    results.append({'video': str(vp), 'video_id': vp.stem, 'success': bool(ok), 'message': msg})
    if ok:
        logger.info(f'SUCCESS: {vp} -> {vp.stem}.wav ({msg})')
    else:
        logger.error(f'FAIL: {vp} -> {msg}')

# Save summary
summary_df = pd.DataFrame(results)
summary_csv = log_dir / 'extraction_summary.csv'
summary_df.to_csv(summary_csv, index=False)
logger.info('--- Extraction summary ---')
logger.info(f'Total videos found: {total}')
logger.info(f'Processed: {len(results)}')
# compute success/failure counts safely
successful = int(summary_df['success'].sum()) if not summary_df.empty else 0
failures = (len(summary_df) - successful) if not summary_df.empty else 0
logger.info(f'Successful exports: {successful}')
logger.info(f'Failures: {failures}')

# Display a small table of failures if any
if (not summary_df.empty) and (successful < len(summary_df)):
    display(summary_df[summary_df['success'] == False].head())
else:
    display(summary_df.head())

2025-11-06 08:27:22,112 - INFO - Found 197 video files to process
Extracting audio:   0%|          | 0/197 [00:01<?, ?it/s]

MoviePy - Writing audio in c:\Users\CelinaB\Documents\FHDW\Semester 5\Projekt Algorithmen\Viralytics\Viralitaetsanalyse\data\processed\audio\top_100_likes_339500_id_7498461301472070954.wav


Extracting audio:   1%|          | 1/197 [00:01<06:12,  1.90s/it]

MoviePy - Done.


Extracting audio:   1%|          | 1/197 [00:03<06:12,  1.90s/it]

...

MoviePy - Writing audio in c:\Users\CelinaB\Documents\FHDW\Semester 5\Projekt Algorithmen\Viralytics\Viralitaetsanalyse\data\processed\audio\normal_9_likes_9707_id_7559684939684400414.wav


Extracting audio: 100%|██████████| 197/197 [06:05<00:00,  1.85s/it]


MoviePy - Done.


2025-11-06 08:33:27,833 - INFO - --- Extraction summary ---
2025-11-06 08:33:27,834 - INFO - Total videos found: 197
2025-11-06 08:33:27,834 - INFO - Processed: 197
2025-11-06 08:33:27,835 - INFO - Successful exports: 197
2025-11-06 08:33:27,836 - INFO - Failures: 0
2025-11-06 08:33:27,834 - INFO - Total videos found: 197
2025-11-06 08:33:27,834 - INFO - Processed: 197
2025-11-06 08:33:27,835 - INFO - Successful exports: 197
2025-11-06 08:33:27,836 - INFO - Failures: 0


,video,video_id,success,message
0,c:\Users\CelinaB\Documents\FHDW\Semester 5\Pro...,top_100_likes_339500_id_7498461301472070954,True,OK
1,c:\Users\CelinaB\Documents\FHDW\Semester 5\Pro...,top_10_likes_2000000_id_7219026508239686954,True,OK
2,c:\Users\CelinaB\Documents\FHDW\Semester 5\Pro...,top_11_likes_2000000_id_7554767993666915615,True,OK
3,c:\Users\CelinaB\Documents\FHDW\Semester 5\Pro...,top_12_likes_1900000_id_7556096007285591351,True,OK
4,c:\Users\CelinaB\Documents\FHDW\Semester 5\Pro...,top_13_likes_1900000_id_7546069423900200247,True,OK
